# 01 - EDA: TBX11K-simplified

Misma estructura que `00_eda_chest_xray.ipynb` (ver ese notebook para la explicación general). Lo que cambia aquí: las etiquetas vienen de un **CSV** (`data.csv`), no de carpetas, y hay una carpeta `test/` que **no tiene ninguna etiqueta** (se explica en la Sección 2).

## 0. Configuración

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "TBX11K-simplified"
DATASET_DIR = Path("../Datasets/tbx11k-simplified")
CSV_PATH = DATASET_DIR / "data.csv"
IMAGES_DIR = DATASET_DIR / "images"
TEST_DIR = DATASET_DIR / "test"

# image_type de origen -> clase objetivo del proyecto
# "sick_but_no_tb" se excluye a propósito: son radiografías ANORMALES pero no de tuberculosis
# (otras patologías) — no son ni Normal ni Tuberculosis, no encajan en ninguna de las 4 clases.
CLASS_MAP = {
    "healthy": "Normal",
    "tb": "Tuberculosis",
}
EXCLUDED_IMAGE_TYPES = {"sick_but_no_tb"}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventario básico
Aquí el inventario parte del CSV, no de listar carpetas — pero se cruza contra los archivos físicos para detectar filas sin imagen o imágenes sin fila.

In [ ]:
raw = pd.read_csv(CSV_PATH)
print(f"Filas en data.csv: {len(raw)}")
raw.head()

In [ ]:
existing_images = {p.name for p in eu.list_images(IMAGES_DIR)}
existing_test = {p.name for p in eu.list_images(TEST_DIR)}
print(f"Archivos físicos en images/: {len(existing_images)}")
print(f"Archivos físicos en test/:   {len(existing_test)}")

missing = raw[~raw["fname"].isin(existing_images)]
orphan = existing_images - set(raw["fname"])
print(f"Filas del CSV sin archivo físico: {len(missing)}")
print(f"Archivos en images/ sin fila en el CSV (huérfanos): {len(orphan)} -> {sorted(orphan)}")

### ⚠️ Hallazgo importante: el CSV tiene una fila POR CAJA DELIMITADORA (bbox), no por imagen

`data.csv` trae varias filas para una misma imagen cuando tiene más de una lesión anotada (columna `bbox`). Si se cuenta "a lo bruto" por filas, Tuberculosis parece tener más imágenes de las que realmente hay. Hay que deduplicar por `fname` **antes** de contar o de mapear clases.

In [ ]:
print(f"Filas totales: {len(raw)}")
print(f"Nombres de archivo únicos: {raw['fname'].nunique()}")
print(f"Filas 'de más' por múltiples cajas en la misma imagen: {raw['fname'].duplicated().sum()}")
print()
print("¿En qué image_type ocurre esto?")
raw.groupby("image_type")["fname"].apply(lambda s: s.duplicated().sum())

In [ ]:
print("Conteo CONTANDO FILAS (incorrecto, infla Tuberculosis):")
display(raw["image_type"].value_counts())
print()
df_unique = raw.drop_duplicates(subset="fname").copy()
print("Conteo real por IMAGEN ÚNICA (correcto):")
df_unique["image_type"].value_counts()

### La carpeta `test/` (3,302 imágenes) — corrección a una suposición anterior

En `01_dataset_preprocessing.ipynb` (notebook de fusión) se asumió que `test/` seguía "la misma convención de prefijo de nombre (`tb*`, `h*`, `s*`)" que `images/`, para poder rescatar TB adicional de ahí por nombre de archivo. **Esto es incorrecto**: los archivos de `test/` se llaman todos `unknown_<n>.png`, sin ningún prefijo de clase, y no existe ningún CSV ni archivo de etiquetas para ellos en este dataset. Es el conjunto de test *sin publicar* del reto original de TBX11K (para el leaderboard de la competencia) — no se puede usar para clasificación supervisada.

**Conclusión: `test/` debe excluirse por completo, y esa parte del notebook 01 hay que corregirla** (no va a encontrar ningún archivo `tb*`/`h*`/`s*` ahí porque no existen).

In [ ]:
import re
prefixes = sorted({re.match(r"[a-zA-Z]+", f).group() for f in existing_test})
print("Prefijos de nombre encontrados en test/:", prefixes)

## 2. Esquema de etiquetas y mapeo a clases objetivo
Se parte de `df_unique` (ya deduplicado por imagen) y se mapea la columna `image_type`. `test/` se descarta por completo (sin etiquetas, sección anterior).

In [ ]:
excluded = df_unique[df_unique["image_type"].isin(EXCLUDED_IMAGE_TYPES)]
print(f"Excluidas por ser 'sick_but_no_tb' (anormales pero no TB, ambiguas para nuestras 4 clases): {len(excluded)}")

df_unique["target_class"] = df_unique["image_type"].map(CLASS_MAP)
unmapped = df_unique[df_unique["target_class"].isna() & ~df_unique["image_type"].isin(EXCLUDED_IMAGE_TYPES)]
if len(unmapped):
    print("image_type sin mapear (revisar CLASS_MAP):", sorted(unmapped["image_type"].unique()))

df = df_unique.dropna(subset=["target_class"]).copy()
df["path"] = df["fname"].apply(lambda f: str(IMAGES_DIR / f))
df["split"] = df["source"]  # 'train' / 'val' tal como vienen en el CSV; no hay 'test' etiquetado
print(f"Total utilizable (Normal + Tuberculosis, sin duplicar por bbox): {len(df)}")

## 3. Conteo real por clase objetivo (por split)

In [ ]:
summary = df.groupby(["split", "target_class"]).size().unstack(fill_value=0)
summary = summary.reindex(columns=sorted(summary.columns))
summary.loc["Total"] = summary.sum()
summary["Total"] = summary.sum(axis=1)
summary.to_csv(OUTPUT_DIR / "conteo_por_clase.csv")
summary

In [ ]:
totales = df["target_class"].value_counts().sort_index()
ax = totales.plot(kind="bar", figsize=(6, 4), color="#55A868")
ax.set_title(f"{DATASET_NAME}: total de imágenes por clase (útiles para el proyecto)")
ax.set_ylabel("Número de imágenes")
for i, v in enumerate(totales):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase.png", dpi=120)
plt.show()

## 4. Calidad e integridad

In [ ]:
probe_results = [eu.probe_image(p) for p in tqdm(df["path"], desc="Verificando imágenes")]
df["width"], df["height"], df["mode"], df["error"] = zip(*probe_results)

corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no abribles: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Modo de color:")
display(df["mode"].value_counts())
print()
print("Dimensiones declaradas en el CSV vs. dimensiones reales leídas:")
print("CSV -> ", df[["image_height", "image_width"]].describe().loc[["min", "max"]].to_dict())
df[["width", "height"]].describe()

### 4.1 Candidatas a excluir por calidad de imagen (mismo criterio que en Chest_x_ray)
Ver `00_eda_chest_xray.ipynb` sección 4.1 para la explicación completa del criterio y la postura ética (se marca para revisión manual, no se borra nada automáticamente).

In [ ]:
calidad = eu.flag_quality_issues(df[df["error"].isna()], desc=DATASET_NAME)
sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
sospechosas["target_class"].value_counts()

In [ ]:
sospechosas[["path", "target_class", "split", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)
eu.show_flagged(sospechosas, save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png")

## 5. Muestra visual por clase

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png")

Y, solo para transparencia, cómo se ven las excluidas `sick_but_no_tb` (anormales pero no TB) que quedaron fuera del mapeo:

In [ ]:
excluded_sample = excluded.copy()
excluded_sample["path"] = excluded_sample["fname"].apply(lambda f: str(IMAGES_DIR / f))
excluded_sample["target_class"] = "sick_but_no_tb (excluida)"
eu.show_samples(excluded_sample, n_per_class=4)

## 6. Riesgo de solapamiento / duplicados
Igual que en Chest_x_ray: hash MD5 por imagen, revisar fuga entre splits, y guardar para el cruce posterior entre TODAS las fuentes.

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
df["phash"] = [eu.phash_file(p) for p in tqdm(df["path"], desc="Hash perceptual")]
n_dupes = df["md5"].duplicated().sum()
print(f"Duplicados exactos dentro de este dataset: {n_dupes}")

splits_por_hash = df.groupby("md5")["split"].nunique()
leak_hashes = splits_por_hash[splits_por_hash > 1].index
print(f"Hashes que aparecen en más de un split (posible fuga train/val): {len(leak_hashes)}")

In [ ]:
df[["path", "md5", "phash", "target_class", "split"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_md5.csv", index=False
)

## 7. Metadata demográfica / poblacional

A diferencia de Chest_x_ray, aquí sí hay algo de metadata en el CSV, aunque no demográfica de paciente (no hay edad/sexo):
- `image_height` / `image_width`: dimensión original declarada (512×512 para todas).
- `source`: split de origen (`train` / `val`).
- `tb_type`: para los casos de Tuberculosis, distingue `active_tb` vs `latent_tb` — un matiz clínico que Chest_x_ray no tiene y que podría mencionarse en Materials and Methods.
- `bbox`: coordenadas de la(s) lesión(es) para los casos de TB — no se usa para clasificación, pero podría servir más adelante para explicabilidad/localización (Grad-CAM comparado contra la caja real).

Sigue sin haber edad, sexo, ni información del equipo/hospital de origen — no se puede documentar la población de este dataset más allá de lo que diga el paper original de TBX11K.

In [ ]:
print("Distribución de tb_type entre las imágenes únicas de Tuberculosis:")
df_unique[df_unique["image_type"] == "tb"]["tb_type"].value_counts()

## 8. Veredicto

- **Hallazgo clave**: el CSV cuenta 1,211 filas de tipo `tb`, pero corresponden a solo **799 imágenes únicas** (412 filas de más por lesiones múltiples en la misma imagen). El notebook de fusión (`01_dataset_preprocessing.ipynb`) asumía implícitamente un número más cercano a 1,200 al fijar `TARGET_PER_CLASS['Tuberculosis']['tbx11k'] = 1200` — el techo real disponible en esta fuente es 799, no 1,200.
- **`test/` (3,302 imágenes) no es utilizable**: no tiene ninguna etiqueta ni sigue la convención de nombre que se había asumido. Hay que corregir esa parte del notebook 01.
- Normal disponible: 3,800 imágenes (de sobra respecto a lo que probablemente se necesite).
- 3,800 imágenes `sick_but_no_tb` correctamente excluidas (anormales pero no TB, no encajan en ninguna de las 4 clases del proyecto).
- Sin metadata demográfica de paciente; sí hay distinción clínica `active_tb` vs `latent_tb` dentro de Tuberculosis, y coordenadas de lesión (`bbox`) reutilizables más adelante para explicabilidad.
- Aporta como techo real: hasta 799 Tuberculosis + hasta 3,800 Normal (probablemente no se necesiten los 3,800, dado que Normal ya tiene volumen de otras fuentes).